In [ ]:
import os
import sys
import time
import copy
import math
import random
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from PIL import Image

In [ ]:
import os
import sys
import time
import copy
import math
import random
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from PIL import Image

In [ ]:
import os
import sys
import time
import copy
import math
import random
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from PIL import Image

In [ ]:
import os
import sys
import time
import copy
import math
import random
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from PIL import Image

In [ ]:
import os
import sys
import time
import copy
import math
import random
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from PIL import Image

In [ ]:
import os
import sys
import time
import copy
import math
import random
import glob
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from PIL import Image

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import copy

In [ ]:
from einops import rearrange
import urllib.request
import warnings

In [ ]:
import random
import numpy as np
import torch
SEED = 67
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
DATA_ROOT = Path('/data/BEN_14k')
METADATA_PATH = DATA_ROOT / 'metadata.parquet'
S1_ROOT = DATA_ROOT / 'BigEarthNet-S1'
S2_ROOT = DATA_ROOT / 'BigEarthNet-S2'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_EPOCHS = 200
S2_CHANNELS = 10

In [ ]:
print('Device:', DEVICE)
for split in ['train', 'validation', 'test']:
    print(split, len(os.listdir(S1_ROOT / split)), len(os.listdir(S2_ROOT / split)))

In [ ]:
metadata = pd.read_parquet(METADATA_PATH)
metadata.head()

In [ ]:
def build_dataframe():
    split_tables = []
    for split in ['train', 'validation', 'test']:
        s1_files = {os.path.splitext(f)[0] for f in os.listdir(S1_ROOT / split) if f.endswith('.tif')}
        s2_files = {os.path.splitext(f)[0] for f in os.listdir(S2_ROOT / split) if f.endswith('.tif')}
        part = metadata[metadata['s1_name'].isin(s1_files) & metadata['patch_id'].isin(s2_files)].copy()
        part['split'] = split
        part['s1_path'] = [S1_ROOT / split / f'{name}.tif' for name in part['s1_name']]
        part['s2_path'] = [S2_ROOT / split / f'{name}.tif' for name in part['patch_id']]
        split_tables.append(part)
    return pd.concat(split_tables, ignore_index=True)

In [ ]:
df = build_dataframe()
print(df['split'].value_counts())
df.head()

In [ ]:
LABELS = sorted({label for labels in df['labels'] for label in labels})
label_to_idx = {label: i for i, label in enumerate(LABELS)}

In [ ]:
class BigEarthNetDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def load_image(self, path):
        try:
            with rasterio.open(path) as src:
                image = src.read(out_shape=(src.count, 224, 224), resampling=Resampling.bilinear).astype(np.float32)
            return image
        except Exception as e:
            raise RuntimeError(f'Error loading image file {path}: {e}')

    def encode_labels(self, labels):
        target = torch.zeros(len(LABELS))
        for label in labels:
            target[label_to_idx[label]] = 1
        return target

    def __getitem__(self, idx):
        sample = self.df.iloc[idx]
        sar = self.load_image(sample.s1_path)
        optical = self.load_image(sample.s2_path)
        sar = np.clip(sar, -25, 5)
        sar = (sar + 25) / 30
        optical = np.clip(optical, 0, 10000)
        optical = optical / 10000.0
        sar = torch.from_numpy(sar)
        optical = torch.from_numpy(optical)
        labels = self.encode_labels(sample.labels)
        return (sar, optical, labels)

In [ ]:
train_dataset = BigEarthNetDataset(df[df['split'] == 'train'])
val_dataset = BigEarthNetDataset(df[df['split'] == 'validation'])
test_dataset = BigEarthNetDataset(df[df['split'] == 'test'])

In [ ]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

In [ ]:
sar, optical, labels = train_dataset[0]
print('SAR:', sar.shape, sar.dtype)
print('Optical:', optical.shape, optical.dtype)
print('Labels:', labels.shape)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(sar[0], cmap='gray')
ax[0].set_title('Sentinel-1')
rgb = optical[[2, 1, 0]].permute(1, 2, 0).numpy()
rgb = np.clip(rgb, 0, 1)
ax[1].imshow(rgb)
ax[1].set_title('Sentinel-2 RGB')
plt.show()

In [ ]:
BATCH_SIZE = 256
NUM_WORKERS = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True)

In [ ]:
sar, optical, labels = next(iter(train_loader))
print('SAR:', sar.shape)
print('Optical:', optical.shape)
print('Labels:', labels.shape)

In [ ]:
print('SAR range:', sar.min().item(), sar.max().item())
print('Optical range:', optical.min().item(), optical.max().item())

In [ ]:
PATCH_SIZE = 16
IMAGE_SIZE = 224
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2
NUM_SHARED_QUERIES = 16
CONTEXT_MASK_RATIO = 0.5
TARGET_MASK_RATIO = 0.5
NUM_VISIBLE = int(NUM_PATCHES * CONTEXT_MASK_RATIO)
NUM_MASKED = int(NUM_PATCHES * TARGET_MASK_RATIO)
EMBED_DIM = 768
PREDICTOR_EMBED_DIM = 384
D_COMMON = 512
D_UNIQUE = EMBED_DIM - D_COMMON
DEPTH = 12
NUM_HEADS = 12
PREDICTOR_DEPTH = 12
PREDICTOR_HEADS = 12

In [ ]:
def generate_disjoint_masks(batch_size, device, num_patches=NUM_PATCHES, context_ratio=CONTEXT_MASK_RATIO, target_ratio=TARGET_MASK_RATIO):
    num_context = int(num_patches * context_ratio)
    num_target = int(num_patches * target_ratio)
    context_visible_indices = []
    target_masked_indices = []
    for _ in range(batch_size):
        perm = torch.randperm(num_patches, device=device)
        ctx_idx = perm[:num_context]
        tgt_idx = perm[num_context:num_context + num_target]
        context_visible_indices.append(ctx_idx)
        target_masked_indices.append(tgt_idx)
    return (torch.stack(context_visible_indices), torch.stack(target_masked_indices))

In [ ]:
visible_idx, masked_idx = generate_disjoint_masks(4, DEVICE)
print('Context visible shape:', visible_idx.shape)
print('Target masked shape:', masked_idx.shape)
overlap = set(visible_idx[0].cpu().tolist()) & set(masked_idx[0].cpu().tolist())
print('Sample 0 overlap count:', len(overlap))
assert len(overlap) == 0, 'Masks must be strictly disjoint!'

In [ ]:
context_mask = torch.zeros(NUM_PATCHES)
context_mask[visible_idx[0].cpu()] = 1.0
target_mask = torch.zeros(NUM_PATCHES)
target_mask[masked_idx[0].cpu()] = 1.0
print('Context visible patches:', context_mask.sum().item())
print('Target masked patches:', target_mask.sum().item())

In [ ]:
def split_visible_masked(tokens, visible_idx, masked_idx):
    B, N, D = tokens.shape
    device = tokens.device
    visible_idx = visible_idx.to(device)
    masked_idx = masked_idx.to(device)
    visible_tokens = torch.gather(tokens, 1, visible_idx.unsqueeze(-1).expand(-1, -1, D))
    target_tokens = torch.gather(tokens, 1, masked_idx.unsqueeze(-1).expand(-1, -1, D))
    return (visible_tokens, target_tokens)

In [ ]:
dummy = torch.randn(2, NUM_PATCHES, EMBED_DIM, device=DEVICE)
v_tok, t_tok = split_visible_masked(dummy, visible_idx[:2], masked_idx[:2])
print('Visible tokens shape:', v_tok.shape)
print('Target tokens shape:', t_tok.shape)

In [ ]:
EMBED_DIM = 768

In [ ]:
class PatchEmbedding(torch.nn.Module):

    def __init__(self, in_channels, embed_dim=EMBED_DIM, patch_size=PATCH_SIZE):
        super().__init__()
        self.proj = torch.nn.Conv2d(in_channels=in_channels, out_channels=embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

In [ ]:
class ModalityStem(torch.nn.Module):

    def __init__(self, in_channels, embed_dim=EMBED_DIM):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, embed_dim=embed_dim)
        self.pos_embed = torch.nn.Parameter(torch.zeros(1, NUM_PATCHES, embed_dim))

    def forward(self, x):
        device = self.pos_embed.device
        x = x.to(device)
        return self.patch_embed(x) + self.pos_embed

In [ ]:
sar_stem = ModalityStem(2).to(DEVICE)

In [ ]:
sar = sar.to(DEVICE)
sar_tokens = sar_stem(sar)

In [ ]:
sar_visible_idx, sar_masked_idx = generate_disjoint_masks(sar_tokens.size(0), sar_tokens.device)
sar_visible, sar_target_tokens = split_visible_masked(sar_tokens, sar_visible_idx, sar_masked_idx)
print('SAR tokens:', sar_tokens.shape)
print('SAR visible:', sar_visible.shape)
print('SAR target tokens:', sar_target_tokens.shape)

In [ ]:
class MLP(torch.nn.Module):

    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or dim * 4
        self.fc1 = torch.nn.Linear(dim, hidden_dim)
        self.act = torch.nn.GELU()
        self.fc2 = torch.nn.Linear(hidden_dim, dim)

    def forward(self, x):
        device = self.fc1.weight.device
        x = x.to(device)
        return self.fc2(self.act(self.fc1(x)))

In [ ]:
mlp = MLP(EMBED_DIM).to(DEVICE)

In [ ]:
class TransformerBlock(torch.nn.Module):

    def __init__(self, embed_dim=EMBED_DIM, num_heads=NUM_HEADS):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(embed_dim)
        self.attn = torch.nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm2 = torch.nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim)

    def forward(self, x):
        device = self.norm1.weight.device
        x = x.to(device)
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = residual + attn_out
        residual = x
        x = self.norm2(x)
        x = self.mlp(x)
        x = residual + x
        return x

In [ ]:
block = TransformerBlock(EMBED_DIM, NUM_HEADS).to(DEVICE)

In [ ]:
class SharedTransformer(torch.nn.Module):

    def __init__(self, depth=DEPTH):
        super().__init__()
        self.blocks = torch.nn.ModuleList([TransformerBlock(EMBED_DIM, NUM_HEADS) for _ in range(depth)])
        self.norm = torch.nn.LayerNorm(EMBED_DIM)

    def forward(self, x):
        if len(self.blocks) > 0:
            x = x.to(self.blocks[0].norm1.weight.device)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return x

In [ ]:
def build_encoder_trunk(depth=DEPTH):
    return SharedTransformer(depth=depth)

In [ ]:
shared_encoder = build_encoder_trunk().to(DEVICE)

In [ ]:
sar = sar.to(DEVICE)
optical = optical.to(DEVICE)
sar_stem = ModalityStem(2).to(DEVICE)
optical_stem = ModalityStem(S2_CHANNELS).to(DEVICE)
shared_encoder = shared_encoder.to(DEVICE)
visible_idx = visible_idx.to(DEVICE)
masked_idx = masked_idx.to(DEVICE)
sar_tokens = sar_stem(sar)
optical_tokens = optical_stem(optical)
sar_visible, _ = split_visible_masked(sar_tokens, visible_idx[:sar_tokens.size(0)], masked_idx[:sar_tokens.size(0)])
optical_visible, _ = split_visible_masked(optical_tokens, visible_idx[:optical_tokens.size(0)], masked_idx[:optical_tokens.size(0)])
sar_features = shared_encoder(sar_visible)
optical_features = shared_encoder(optical_visible)
print(sar_features.shape)
print(optical_features.shape)

In [ ]:
print('Transformer depth:', len(shared_encoder.blocks))
params = sum((p.numel() for p in shared_encoder.parameters()))
print(f'Parameters: {params:,}')

In [ ]:
with torch.no_grad():
    x = torch.randn(1, NUM_VISIBLE, EMBED_DIM, device=DEVICE)
    y = shared_encoder(x)
print(x.shape)
print(y.shape)

In [ ]:
target_encoder = copy.deepcopy(shared_encoder).to(DEVICE)
for param in target_encoder.parameters():
    param.requires_grad = False
target_encoder.eval()

In [ ]:
EMA_MOMENTUM = 0.996

In [ ]:
@torch.no_grad()
def update_target_model(model, momentum=EMA_MOMENTUM):
    for online_param, target_param in zip(model.online_encoder.parameters(), model.target_encoder.parameters()):
        target_param.data.mul_(momentum)
        target_param.data.add_(online_param.data, alpha=1.0 - momentum)
    for online_param, target_param in zip(model.sar_stem.parameters(), model.target_sar_stem.parameters()):
        target_param.data.mul_(momentum)
        target_param.data.add_(online_param.data, alpha=1.0 - momentum)
    for online_param, target_param in zip(model.optical_stem.parameters(), model.target_optical_stem.parameters()):
        target_param.data.mul_(momentum)
        target_param.data.add_(online_param.data, alpha=1.0 - momentum)

In [ ]:
online_params = sum((p.numel() for p in shared_encoder.parameters()))
target_params = sum((p.numel() for p in target_encoder.parameters()))
print(online_params)
print(target_params)

In [ ]:
with torch.no_grad():
    target_features = target_encoder(sar_target_tokens)
print(target_features.shape)

In [ ]:
PREDICTOR_DEPTH = 12

In [ ]:
class DDDXJEPAPredictorBlock(nn.Module):

    def __init__(self, dim=PREDICTOR_EMBED_DIM, num_heads=PREDICTOR_HEADS):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.self_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.cross_attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm3 = nn.LayerNorm(dim)
        self.mlp = MLP(dim)

    def forward(self, queries, opposite_context):
        device = self.norm1.weight.device
        queries = queries.to(device)
        opposite_context = opposite_context.to(device)
        res = queries
        x = self.norm1(queries)
        x, _ = self.self_attn(x, x, x, need_weights=False)
        x = res + x
        res = x
        x = self.norm2(x)
        x, _ = self.cross_attn(x, opposite_context, opposite_context, need_weights=False)
        x = res + x
        res = x
        x = self.norm3(x)
        x = self.mlp(x)
        x = res + x
        return x

In [ ]:
predictor_block = DDDXJEPAPredictorBlock().to(DEVICE)

In [ ]:
queries_dummy = torch.randn(2, NUM_SHARED_QUERIES + NUM_MASKED, PREDICTOR_EMBED_DIM, device=DEVICE)
context_dummy = torch.randn(2, NUM_VISIBLE, PREDICTOR_EMBED_DIM, device=DEVICE)
out_dummy = predictor_block(queries_dummy, context_dummy)
print('Predictor block output shape:', out_dummy.shape)

In [ ]:
class DDXJEPAPredictor(nn.Module):

    def __init__(self, in_dim=EMBED_DIM, pred_dim=PREDICTOR_EMBED_DIM, depth=PREDICTOR_DEPTH, num_heads=PREDICTOR_HEADS, num_patches=NUM_PATCHES):
        super().__init__()
        self.proj_in_context = nn.Linear(in_dim, pred_dim)
        self.proj_out = nn.Linear(pred_dim, in_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, pred_dim))
        self.pos_embed_sar = nn.Parameter(torch.zeros(1, num_patches, pred_dim))
        self.pos_embed_optical = nn.Parameter(torch.zeros(1, num_patches, pred_dim))
        self.blocks = nn.ModuleList([DDDXJEPAPredictorBlock(dim=pred_dim, num_heads=num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(pred_dim)
        nn.init.trunc_normal_(self.mask_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed_sar, std=0.02)
        nn.init.trunc_normal_(self.pos_embed_optical, std=0.02)

    def forward(self, context_tokens, opposite_context, masked_indices, shared_queries, target_modality='sar'):
        B = context_tokens.size(0)
        device = context_tokens.device
        N_target = masked_indices.size(1)
        if masked_indices.size(0) != B:
            if masked_indices.size(0) > B:
                masked_indices = masked_indices[:B]
            else:
                reps = math.ceil(B / masked_indices.size(0))
                masked_indices = masked_indices.repeat(reps, 1)[:B]
        opposite_context = opposite_context.to(device)
        masked_indices = masked_indices.to(device)
        shared_queries = shared_queries.to(device)
        ctx_proj = self.proj_in_context(opposite_context)
        mask_tokens = self.mask_token.to(device).expand(B, N_target, PREDICTOR_EMBED_DIM)
        pos_table = (self.pos_embed_sar if target_modality == 'sar' else self.pos_embed_optical).to(device)
        target_pos = torch.gather(pos_table.expand(B, NUM_PATCHES, PREDICTOR_EMBED_DIM), 1, masked_indices.unsqueeze(-1).expand(-1, -1, PREDICTOR_EMBED_DIM))
        routed_mask_tokens = mask_tokens + target_pos
        s_expanded = shared_queries.expand(B, -1, -1)
        num_queries = s_expanded.size(1)
        queries = torch.cat([s_expanded, routed_mask_tokens], dim=1)
        x = queries
        for block in self.blocks:
            x = block(x, ctx_proj)
        x = self.norm(x)
        target_preds_384 = x[:, num_queries:, :]
        predicted_targets = self.proj_out(target_preds_384)
        return predicted_targets

In [ ]:
ddxjepa_predictor = DDXJEPAPredictor().to(DEVICE)
shared_queries_dummy = nn.Parameter(torch.randn(1, NUM_SHARED_QUERIES, PREDICTOR_EMBED_DIM, device=DEVICE))

In [ ]:
pred_dummy = ddxjepa_predictor(sar_features, optical_features, masked_idx[:sar_features.size(0)], shared_queries_dummy, target_modality='optical')
print('Predicted targets shape:', pred_dummy.shape)

In [ ]:
sar_visible_idx, sar_masked_idx = generate_disjoint_masks(sar_tokens.size(0), sar_tokens.device)
optical_visible_idx, optical_masked_idx = generate_disjoint_masks(optical_tokens.size(0), optical_tokens.device)
sar_visible, sar_target_tokens = split_visible_masked(sar_tokens, sar_visible_idx, sar_masked_idx)
optical_visible, optical_target_tokens = split_visible_masked(optical_tokens, optical_visible_idx, optical_masked_idx)

In [ ]:
sar_context = shared_encoder(sar_visible)
optical_context = shared_encoder(optical_visible)
with torch.no_grad():
    sar_target_full = target_encoder(sar_tokens)
    optical_target_full = target_encoder(optical_tokens)
    _, sar_target = split_visible_masked(sar_target_full, sar_visible_idx, sar_masked_idx)
    _, optical_target = split_visible_masked(optical_target_full, optical_visible_idx, optical_masked_idx)

In [ ]:
sar_to_optical_pred = ddxjepa_predictor(sar_context, optical_context, optical_masked_idx, shared_queries_dummy, target_modality='optical')
optical_to_sar_pred = ddxjepa_predictor(optical_context, sar_context, sar_masked_idx, shared_queries_dummy, target_modality='sar')

In [ ]:
print('SAR to Optical prediction shape:', sar_to_optical_pred.shape)
print('Optical to SAR prediction shape:', optical_to_sar_pred.shape)
print('Optical target shape:', optical_target.shape)
print('SAR target shape:', sar_target.shape)

In [ ]:
print('Cosine similarity (SAR->Optical target):', F.cosine_similarity(sar_to_optical_pred.flatten(1), optical_target.flatten(1), dim=1).mean().item())
print('Cosine similarity (Optical->SAR target):', F.cosine_similarity(optical_to_sar_pred.flatten(1), sar_target.flatten(1), dim=1).mean().item())

In [ ]:
RETRIEVAL_EVAL = True

In [ ]:
class MeanPooling(nn.Module):

    def __init__(self, dim):
        super().__init__()

    def forward(self, tokens):
        return tokens.mean(dim=1)

def compute_latent_embeddings(context_tokens):
    pool = MeanPooling(EMBED_DIM).to(context_tokens.device)
    emb = pool(context_tokens)
    return F.normalize(emb, dim=-1)

In [ ]:
e_sar = compute_latent_embeddings(sar_context)
e_optical = compute_latent_embeddings(optical_context)

In [ ]:
print('SAR latent embedding shape:', e_sar.shape)
print('Optical latent embedding shape:', e_optical.shape)

In [ ]:
print('SAR embedding norm:', torch.norm(e_sar, dim=1).mean().item())
print('Optical embedding norm:', torch.norm(e_optical, dim=1).mean().item())

In [ ]:
cos_sim = torch.sum(e_sar * e_optical, dim=1).mean().item()
print('Mean cross-modal latent cosine similarity:', cos_sim)

In [ ]:
def compute_cross_retrieval_matrix(sar_latents, optical_latents):
    return torch.matmul(sar_latents, optical_latents.T)

@torch.no_grad()
def evaluate_retrieval_f1(model, val_loader, device=DEVICE, k_list=[5, 10], d_common=D_COMMON):
    model.eval()
    sar_full, opt_full, sar_common, opt_common, labels_list = ([], [], [], [], [])
    for batch in val_loader:
        sar = batch[0].to(device)
        optical = batch[1].to(device)
        lbls = batch[2]
        sar_tokens = model.online_encoder(model.sar_stem(sar))
        opt_tokens = model.online_encoder(model.optical_stem(optical))
        z_sar = model.mean_pool(sar_tokens)
        z_opt = model.mean_pool(opt_tokens)
        sar_full.append(F.normalize(z_sar, dim=-1).cpu())
        opt_full.append(F.normalize(z_opt, dim=-1).cpu())
        sar_common.append(F.normalize(z_sar[:, :d_common], dim=-1).cpu())
        opt_common.append(F.normalize(z_opt[:, :d_common], dim=-1).cpu())
        labels_list.append(lbls.cpu())
    Z_sar_full, Z_opt_full = (torch.cat(sar_full, dim=0), torch.cat(opt_full, dim=0))
    Z_sar_common, Z_opt_common = (torch.cat(sar_common, dim=0), torch.cat(opt_common, dim=0))
    Y = torch.cat(labels_list, dim=0).float()
    N = Y.size(0)
    if N < 2:
        return {}
    Y_sum = Y.sum(dim=1, keepdim=True).clamp(min=1.0)
    intersection = torch.matmul(Y, Y.T)
    graded_p_mat = intersection / Y_sum.T
    graded_r_mat = intersection / Y_sum
    results = {}
    tasks = [('S1->S2', Z_sar_common, Z_opt_common, False), ('S2->S1', Z_opt_common, Z_sar_common, False), ('S1->S1', Z_sar_full, Z_sar_full, True), ('S2->S2', Z_opt_full, Z_opt_full, True)]
    for task_name, Q, G, exclude_self in tasks:
        sim_mat = torch.matmul(Q, G.T)
        if exclude_self:
            sim_mat.fill_diagonal_(-float('inf'))
        for K in k_list:
            topk_idx = sim_mat.topk(min(K, N - 1 if exclude_self else N), dim=1).indices
            p_k = torch.gather(graded_p_mat, 1, topk_idx).mean(dim=1)
            r_k = torch.gather(graded_r_mat, 1, topk_idx).mean(dim=1)
            f1_k = torch.where(p_k + r_k > 0, 2 * p_k * r_k / (p_k + r_k), torch.zeros_like(p_k)).mean().item() * 100.0
            results[f'{task_name}_F1@{K}'] = f1_k
    return results

In [ ]:
sim_matrix = compute_cross_retrieval_matrix(e_sar, e_optical)

In [ ]:
print('Retrieval similarity matrix shape:', sim_matrix.shape)

In [ ]:
top1_acc = (sim_matrix.argmax(dim=1) == torch.arange(sim_matrix.size(0), device=DEVICE)).float().mean().item()
print(f'Top-1 retrieval accuracy on batch: {top1_acc * 100:.2f}%')

In [ ]:
LAMBDA_PSA = 0.1
LAMBDA_INV = 1.0
LAMBDA_VAR = 25.0
LAMBDA_COV = 25.0
LAMBDA_EXCL = 5.0
LAMBDA_VAR_MIN = 5.0
LAMBDA_COV_MIN = 2.0
LAMBDA_EXCL_MIN = 1.0

In [ ]:
class DDXJEPAPSALoss(nn.Module):

    def __init__(self, embed_dim=EMBED_DIM, eps=0.0001):
        super().__init__()
        self.eps = eps
        self.M_weight = nn.Parameter(torch.eye(embed_dim) * 0.1)

    def forward(self, pred, target):
        device = pred.device
        target = target.to(device)
        delta = pred - target
        M_w = self.M_weight.to(device)
        W_delta = torch.matmul(delta, M_w)
        quad_form = torch.sum(W_delta ** 2, dim=-1) + self.eps * torch.sum(delta ** 2, dim=-1)
        quad_form_clamped = torch.clamp(quad_form, min=0.0, max=10000.0)
        psa_loss = torch.mean(torch.sqrt(quad_form_clamped + 1e-06))
        return psa_loss

In [ ]:
psa_loss_fn = DDXJEPAPSALoss(embed_dim=EMBED_DIM).to(DEVICE)
loss_psa_test = psa_loss_fn(sar_to_optical_pred, optical_target)

In [ ]:
print('PSA loss test:', loss_psa_test.item())

In [ ]:
def split_common_unique(z, d_common=D_COMMON):
    return (z[:, :d_common], z[:, d_common:])

def cross_covariance_penalty(A, B):
    N = A.size(0)
    A_c = A - A.mean(dim=0, keepdim=True)
    B_c = B - B.mean(dim=0, keepdim=True)
    cross_cov = A_c.T @ B_c / (N - 1)
    return cross_cov.pow(2).sum() / (A.size(1) * B.size(1))

def decoupled_vicreg_loss(z_a, z_b, d_common=D_COMMON, lambda_inv=LAMBDA_INV, lambda_var=LAMBDA_VAR, lambda_cov=LAMBDA_COV, lambda_excl=LAMBDA_EXCL):
    Z_a = z_a.mean(dim=1) if z_a.ndim == 3 else z_a
    Z_b = z_b.mean(dim=1) if z_b.ndim == 3 else z_b
    c_a, u_a = split_common_unique(Z_a, d_common)
    c_b, u_b = split_common_unique(Z_b, d_common)
    sim_loss = F.mse_loss(c_a, c_b)

    def variance_loss(Z):
        std_Z = torch.sqrt(Z.var(dim=0) + 0.0001)
        return torch.mean(F.relu(1.0 - std_Z))
    var_loss = 0.25 * (variance_loss(c_a) + variance_loss(c_b) + variance_loss(u_a) + variance_loss(u_b))

    def covariance_loss(Z):
        B_, D = Z.shape
        Zc = Z - Z.mean(dim=0, keepdim=True)
        cov = Zc.T @ Zc / (B_ - 1)
        off_diag = cov.pow(2).sum() - cov.diagonal().pow(2).sum()
        return off_diag / (D * D)
    cov_loss = 0.25 * (covariance_loss(c_a) + covariance_loss(c_b) + covariance_loss(u_a) + covariance_loss(u_b))
    excl_loss = (cross_covariance_penalty(c_a, u_a) + cross_covariance_penalty(c_b, u_b) + cross_covariance_penalty(u_a, u_b)) / 3.0
    total = lambda_inv * sim_loss + lambda_var * var_loss + lambda_cov * cov_loss + lambda_excl * excl_loss
    return (total, sim_loss, var_loss, cov_loss, excl_loss)

def vicreg_loss(z_a, z_b, **kwargs):
    return decoupled_vicreg_loss(z_a, z_b, **kwargs)[:4]

In [ ]:
loss_vicreg_test, loss_inv_t, loss_var_t, loss_cov_t, loss_excl_t = decoupled_vicreg_loss(sar_context, optical_context)

In [ ]:
print(f'VICReg Total: {loss_vicreg_test.item():.4f} | Inv: {loss_inv_t.item():.4f} | Var: {loss_var_t.item():.4f} | Cov: {loss_cov_t.item():.4f} | Excl: {loss_excl_t.item():.4f}')

In [ ]:
print('VICReg loss:', loss_vicreg_test.item())

In [ ]:
cosine_sim_context = F.cosine_similarity(e_sar, e_optical, dim=1).mean().item()
print('Mean context cosine similarity:', cosine_sim_context)

In [ ]:
def l2_prediction_loss(prediction, target):
    return F.mse_loss(prediction, target)

In [ ]:
loss_l2_sar_opt = l2_prediction_loss(sar_to_optical_pred, optical_target)
loss_l2_opt_sar = l2_prediction_loss(optical_to_sar_pred, sar_target)
loss_l2_total = 0.5 * (loss_l2_sar_opt + loss_l2_opt_sar)

In [ ]:
loss_psa_sar_opt = psa_loss_fn(sar_to_optical_pred, optical_target)
loss_psa_opt_sar = psa_loss_fn(optical_to_sar_pred, sar_target)
loss_psa_total = 0.5 * (loss_psa_sar_opt + loss_psa_opt_sar)
loss_pred_total = loss_l2_total + LAMBDA_PSA * loss_psa_total

In [ ]:
print(f'L2 Loss: {loss_l2_total.item():.4f} | PSA Loss: {loss_psa_total.item():.4f} | Total Pred: {loss_pred_total.item():.4f}')

In [ ]:
total_loss = loss_pred_total + loss_vicreg_test
print(f'Total DX-JEPA Loss: {total_loss.item():.4f}')

In [ ]:
total_loss.backward()
print('Backward pass successful for DX-JEPA loss.')

In [ ]:
shared_grad = 0.0
for p in shared_encoder.parameters():
    if p.grad is not None:
        shared_grad += p.grad.norm().item()
print('Shared encoder grad norm:', shared_grad)

In [ ]:
class DXJEPA(nn.Module):

    def __init__(self, embed_dim=EMBED_DIM, depth=DEPTH, predictor_depth=PREDICTOR_DEPTH, num_heads=NUM_HEADS, num_queries=NUM_SHARED_QUERIES):
        super().__init__()
        self.sar_stem = ModalityStem(in_channels=2)
        self.optical_stem = ModalityStem(in_channels=S2_CHANNELS)
        self.online_encoder = build_encoder_trunk(depth=depth)
        self.shared_queries = nn.Parameter(torch.randn(1, num_queries, PREDICTOR_EMBED_DIM) * 0.02)
        self.predictor = DDXJEPAPredictor(in_dim=embed_dim, pred_dim=PREDICTOR_EMBED_DIM, depth=predictor_depth, num_heads=PREDICTOR_HEADS)
        self.psa_loss_fn = DDXJEPAPSALoss(embed_dim=embed_dim)
        self.mean_pool = MeanPooling(embed_dim)

    def forward(self, sar, optical):
        B = sar.size(0)
        device = sar.device
        sar_context_visible_idx, sar_target_masked_idx = generate_disjoint_masks(B, device)
        optical_context_visible_idx, optical_target_masked_idx = generate_disjoint_masks(B, device)
        sar_tokens = self.sar_stem(sar)
        optical_tokens = self.optical_stem(optical)
        sar_visible, _ = split_visible_masked(sar_tokens, sar_context_visible_idx, sar_target_masked_idx)
        optical_visible, _ = split_visible_masked(optical_tokens, optical_context_visible_idx, optical_target_masked_idx)
        sar_context = self.online_encoder(sar_visible)
        optical_context = self.online_encoder(optical_visible)
        with torch.no_grad():
            sar_target_full = self.online_encoder(sar_tokens)
            opt_target_full = self.online_encoder(optical_tokens)
            _, sar_target = split_visible_masked(sar_target_full, sar_context_visible_idx, sar_target_masked_idx)
            _, opt_target = split_visible_masked(opt_target_full, optical_context_visible_idx, optical_target_masked_idx)
            sar_target = sar_target.detach()
            opt_target = opt_target.detach()
        sar_to_opt_pred = self.predictor(context_tokens=sar_context, opposite_context=optical_context, masked_indices=optical_target_masked_idx, shared_queries=self.shared_queries, target_modality='optical')
        opt_to_sar_pred = self.predictor(context_tokens=optical_context, opposite_context=sar_context, masked_indices=sar_target_masked_idx, shared_queries=self.shared_queries, target_modality='sar')
        return {'sar_context': sar_context, 'optical_context': optical_context, 'sar_target': sar_target, 'optical_target': opt_target, 'sar_to_opt_pred': sar_to_opt_pred, 'opt_to_sar_pred': opt_to_sar_pred, 'sar_context_visible_idx': sar_context_visible_idx, 'sar_target_masked_idx': sar_target_masked_idx, 'optical_context_visible_idx': optical_context_visible_idx, 'optical_target_masked_idx': optical_target_masked_idx}

In [ ]:
model = DXJEPA()
model.to(DEVICE)

In [ ]:
def model_memory_gb(model):
    total = sum((p.numel() * p.element_size() for p in model.parameters()))
    total += sum((b.numel() * b.element_size() for b in model.buffers()))
    return total / 1024 ** 3
print(f'Model size: {model_memory_gb(model):.2f} GB')

In [ ]:
total_params = sum((p.numel() for p in model.parameters()))
trainable_params = sum((p.numel() for p in model.parameters() if p.requires_grad))
print(f'Total Parameters: {total_params:,}')
print(f'Trainable Parameters: {trainable_params:,}')

In [ ]:
import gc
del shared_encoder, target_encoder
del ddxjepa_predictor
del sar_stem, optical_stem
del sar_tokens, optical_tokens
del sar_visible, optical_visible
del sar_context, optical_context
del sar_target, optical_target
del sar_to_optical_pred, optical_to_sar_pred
del sar_visible_idx, sar_masked_idx
del optical_visible_idx, optical_masked_idx
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
model = DXJEPA().to(DEVICE)

In [ ]:
batch = next(iter(train_loader))
sar = batch[0].to(DEVICE)
optical = batch[1].to(DEVICE)
labels = batch[2].to(DEVICE)
outputs = model(sar, optical)

In [ ]:
for key, value in outputs.items():
    if torch.is_tensor(value):
        print(f'{key:25}', tuple(value.shape))

In [ ]:
import gc
try:
    del batch, sar, optical, labels, outputs
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
LEARNING_RATE = 0.0003
WEIGHT_DECAY = 0.04

In [ ]:
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

In [ ]:
warmup_epochs = 15
if NUM_EPOCHS > warmup_epochs:
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs)
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - warmup_epochs, eta_min=1e-06)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])
else:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-06)

In [ ]:
amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
use_scaler = amp_dtype == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)

In [ ]:
def compute_loss(outputs, model, lambda_inv=LAMBDA_INV, lambda_var=LAMBDA_VAR, lambda_cov=LAMBDA_COV, lambda_excl=LAMBDA_EXCL):
    sar_to_opt_pred = outputs['sar_to_opt_pred']
    opt_to_sar_pred = outputs['opt_to_sar_pred']
    sar_target = outputs['sar_target']
    opt_target = outputs['optical_target']
    sar_context = outputs['sar_context']
    optical_context = outputs['optical_context']
    l2_sar_opt = F.mse_loss(sar_to_opt_pred, opt_target)
    l2_opt_sar = F.mse_loss(opt_to_sar_pred, sar_target)
    loss_l2 = 0.5 * (l2_sar_opt + l2_opt_sar)
    psa_sar_opt = model.psa_loss_fn(sar_to_opt_pred, opt_target)
    psa_opt_sar = model.psa_loss_fn(opt_to_sar_pred, sar_target)
    loss_psa = 0.5 * (psa_sar_opt + psa_opt_sar)
    loss_pred = loss_l2 + LAMBDA_PSA * loss_psa
    loss_vicreg, sim_loss, var_loss, cov_loss, excl_loss = decoupled_vicreg_loss(model.mean_pool(sar_context), model.mean_pool(optical_context), lambda_inv=lambda_inv, lambda_var=lambda_var, lambda_cov=lambda_cov, lambda_excl=lambda_excl)
    total_loss = loss_pred + loss_vicreg
    return {'total_loss': total_loss, 'loss_l2': loss_l2, 'loss_psa': loss_psa, 'loss_pred': loss_pred, 'loss_vicreg': loss_vicreg, 'loss_inv': sim_loss, 'loss_var': var_loss, 'loss_cov': cov_loss, 'loss_excl': excl_loss}

In [ ]:
from tqdm.auto import tqdm
import math
use_amp = torch.cuda.is_available()
amp_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
use_scaler = use_amp and amp_dtype == torch.float16
num_batches = max(1, len(train_loader))
total_steps = NUM_EPOCHS * num_batches
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    running_l2 = 0.0
    running_psa = 0.0
    running_vicreg = 0.0
    pbar = tqdm(train_loader, desc='Epoch %d/%d' % (epoch + 1, NUM_EPOCHS), leave=True)
    for step, batch in enumerate(pbar):
        sar_b = batch[0].to(DEVICE)
        optical_b = batch[1].to(DEVICE)
        optimizer.zero_grad()
        current_step = epoch * num_batches + step
        cosine_decay = math.cos(math.pi * current_step / total_steps) * 0.5 + 0.5
        sched_lambda_var = LAMBDA_VAR_MIN + (LAMBDA_VAR - LAMBDA_VAR_MIN) * cosine_decay
        sched_lambda_cov = LAMBDA_COV_MIN + (LAMBDA_COV - LAMBDA_COV_MIN) * cosine_decay
        sched_lambda_excl = LAMBDA_EXCL_MIN + (LAMBDA_EXCL - LAMBDA_EXCL_MIN) * cosine_decay
        with torch.amp.autocast(device_type=DEVICE.type, dtype=amp_dtype, enabled=use_amp):
            outputs = model(sar_b, optical_b)
            loss_dict = compute_loss(outputs, model, lambda_var=sched_lambda_var, lambda_cov=sched_lambda_cov, lambda_excl=sched_lambda_excl)
            loss = loss_dict['total_loss']
        if use_scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        running_loss += loss.item()
        running_l2 += loss_dict['loss_l2'].item()
        running_psa += loss_dict['loss_psa'].item()
        running_vicreg += loss_dict['loss_vicreg'].item()
        lr_val = optimizer.param_groups[0]['lr']
        pbar.set_postfix(loss='%.4f' % loss.item(), l2='%.4f' % (running_l2 / (step + 1)), psa='%.4f' % (running_psa / (step + 1)), vic='%.4f' % (running_vicreg / (step + 1)), lr='%.2e' % lr_val)
    scheduler.step()
    steps_in_epoch = max(1, step + 1 if 'step' in locals() else num_batches)
    epoch_loss = running_loss / steps_in_epoch
    epoch_l2 = running_l2 / steps_in_epoch
    epoch_psa = running_psa / steps_in_epoch
    epoch_vicreg = running_vicreg / steps_in_epoch
    print()
    print('Epoch %3d/%d | Total Loss: %.4f | L2 Loss: %.4f | PSA Loss: %.4f | VICReg Loss: %.4f' % (epoch + 1, NUM_EPOCHS, epoch_loss, epoch_l2, epoch_psa, epoch_vicreg))
    if (epoch + 1) % 10 == 0:
        print()
        print('Validation Retrieval Evaluation (Epoch %d) ---' % (epoch + 1))
        val_metrics = evaluate_retrieval_f1(model, val_loader, device=DEVICE)
        if val_metrics:
            print('S1->S2 (SAR->Optical)  | F1@5: %.2f%% | F1@10: %.2f%%' % (val_metrics.get('S1->S2_F1@5', 0.0), val_metrics.get('S1->S2_F1@10', 0.0)))
            print('S2->S1 (Optical->SAR)  | F1@5: %.2f%% | F1@10: %.2f%%' % (val_metrics.get('S2->S1_F1@5', 0.0), val_metrics.get('S2->S1_F1@10', 0.0)))
            print('S1->S1 (SAR->SAR)      | F1@5: %.2f%% | F1@10: %.2f%%' % (val_metrics.get('S1->S1_F1@5', 0.0), val_metrics.get('S1->S1_F1@10', 0.0)))
            print('S2->S2 (Optical->Opt)  | F1@5: %.2f%% | F1@10: %.2f%%' % (val_metrics.get('S2->S2_F1@5', 0.0), val_metrics.get('S2->S2_F1@10', 0.0)))
            print()
    if (epoch + 1) % 50 == 0:
        checkpoint = {'epoch': epoch + 1, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(), 'loss': epoch_loss, 'loss_l2': epoch_l2, 'loss_psa': epoch_psa, 'loss_vicreg': epoch_vicreg}
        checkpoint_path = 'dxjepa_epoch_%d.pth' % (epoch + 1)
        torch.save(checkpoint, checkpoint_path)
        print('Checkpoint saved: %s' % checkpoint_path)

In [ ]:
model_save_path = 'dxjepa_model.pth'
torch.save(model.state_dict(), model_save_path)
print(f'DX-JEPA model successfully saved to {model_save_path}')